In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# for better visuals
sns.set(style="whitegrid")

In [12]:
df = pd.read_csv(r"E:\Supervised Project\data\data.csv", encoding='ISO-8859-1')


In [13]:
import pandas as pd

# if you saved rfm earlier, load it, else recreate
df = pd.read_csv("../data/data.csv", encoding='ISO-8859-1')

df = df.dropna(subset=['CustomerID'])
df['CustomerID'] = df['CustomerID'].astype(int)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.reset_index(inplace=True)

In [14]:
rfm['HighValue'] = (rfm['Monetary'] > rfm['Monetary'].median()).astype(int)

In [15]:
from sklearn.model_selection import train_test_split

X = rfm[['Recency','Frequency','Monetary']]
y = rfm['HighValue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [17]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [19]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def evaluate(y_test, y_pred, model_name):
    print(f"----- {model_name} -----")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Report:\n", classification_report(y_test, y_pred))

evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_knn, "KNN")
evaluate(y_test, y_pred_rf, "Random Forest")

----- Logistic Regression -----
Accuracy: 0.9988479262672811
Confusion Matrix:
 [[429   0]
 [  1 438]]
Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       429
           1       1.00      1.00      1.00       439

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868

----- KNN -----
Accuracy: 0.9942396313364056
Confusion Matrix:
 [[428   1]
 [  4 435]]
Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99       429
           1       1.00      0.99      0.99       439

    accuracy                           0.99       868
   macro avg       0.99      0.99      0.99       868
weighted avg       0.99      0.99      0.99       868

----- Random Forest -----
Accuracy: 0.9988479262672811
Confusion Matrix:
 [[429   0]
 [  1 438]]
Report:
               precision    recal

In [20]:
import joblib

joblib.dump(rf, "../models/classifier.pkl")

['../models/classifier.pkl']